In [0]:
%sql
USE CATALOG brazil_car_fleet;

USE SCHEMA silver;

In [0]:
# -----------------------------------------------------------------------------
# 1. Imports e definição da função principal
# -----------------------------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Mapeamento mês PT → número (via when/otherwise)
def mes_para_numero(col):
    return (
        F.when(F.lower(col) == "janeiro",   1)
         .when(F.lower(col) == "fevereiro", 2)
         .when(F.lower(col) == "marco",     3)  # sem acento pois regex pode variar
         .when(F.lower(col) == "março",     3)
         .when(F.lower(col) == "maro",      3)
         .when(F.lower(col) == "abril",     4)
         .when(F.lower(col) == "maio",      5)
         .when(F.lower(col) == "junho",     6)
         .when(F.lower(col) == "julho",     7)
         .when(F.lower(col) == "agosto",    8)
         .when(F.lower(col) == "setembro",  9)
         .when(F.lower(col) == "outubro",  10)
         .when(F.lower(col) == "novembro", 11)
         .when(F.lower(col) == "dezembro", 12)
         .otherwise(None)
    )

In [0]:
# -----------------------------------------------------------------------------
# 2. Cria DF contendo apenas a coluna nm_file da tabela bronze.brazil_car_fleet
# -----------------------------------------------------------------------------

df_nm_file = (
    spark.table("bronze.brazil_car_fleet")
    .select("nm_file")
    .distinct()
)


In [0]:
# -----------------------------------------------------------------------------
# 3. Extrai nome do mes da nomenclatura do arquivo original
# -----------------------------------------------------------------------------

df_dim_data = (df_nm_file    
    .withColumn(
        "nm_mes",
        F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 1)
    )
)

In [0]:
# -----------------------------------------------------------------------------
# 4. Obtem as outras colunas (nr_ano, dt_referencia, id_data) a partir
#    da nomenclatura do arquivo original
# -----------------------------------------------------------------------------

df_dim_data = (
    df_dim_data    
    .withColumn(
        "nr_ano",
        F.regexp_extract(F.lower("nm_file"), r"combustivel_([A-Za-záéíóúãõêç]+)_(\d{4})", 2).cast("int")
    )
    .withColumn("nr_mes", mes_para_numero(F.col("nm_mes")))
    .withColumn(
        "dt_referencia",
        F.to_date(F.concat_ws("-", F.col("nr_ano"), F.col("nr_mes"), F.lit("01")), "yyyy-M-d")
    )
    .withColumn(
        "id_data",
        (F.col("nr_ano") * 100 + F.col("nr_mes")).cast("int")  # ex: 202404
    )
    .select("id_data", "nm_mes", "nr_mes", "nr_ano", "dt_referencia")
    .orderBy("id_data")
)

In [0]:
# -----------------------------------------------------------------------------
# 4. Transforma o nome do mês para o padrão PT (ex: maro/marco → março)
#    - identificado que o mês de março vem como "maro"
# -----------------------------------------------------------------------------

df_dim_data = df_dim_data.withColumn(
    "nm_mes",
    F.when(F.col("nm_mes").isin(["maro", "marco"]), "março").otherwise(F.col("nm_mes"))
)

In [0]:
# Auditoria rápida — checar se algum mês ficou nulo
nulos = df_dim_data.filter(F.col("nr_mes").isNull()).count()
if nulos > 0:
    print(f"⚠️  {nulos} registro(s) com nr_mes nulo — verificar grafia no nm_file:")
    df_dim_data.filter(F.col("nr_mes").isNull()).show(truncate=False)

In [0]:
# -----------------------------------------------------------------------------
# 5. Salvar na Silver como Delta
# -----------------------------------------------------------------------------

(
    df_dim_data
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_data")
)

print("silver.dim_data salva com sucesso.")
df_dim_data.show(truncate=False)